In [ ]:
# [CELL 1] CORE DEPENDENCIES & REPRODUCIBILITY
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import joblib
import warnings
import pmdarima

# Traditional Statistics & Machine Learning Modules
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error
from sklearn.linear_model import LinearRegression
import pmdarima as pm
from statsmodels.tsa.arima.model import ARIMA

# Suppress convergence warnings during rolling ARIMA fits
warnings.filterwarnings("ignore")

# Guarantee absolute reproducibility across runs
os.environ['PYTHONHASHSEED'] = '42'
np.random.seed(42)

print("All foundational traditional statistical and pipeline dependencies successfully loaded.")

In [ ]:
# [CELL 2] GLOBAL EXPERIMENT CONFIGURATION
CONFIG = {
    # Dataset Configuration
    "FILE_PATH": "File_Path/To/Your/Dataset.csv"  # Path to the CSV dataset

    # Model Selection Architecture
    # Options: 'ARIMA', 'LINEAR_REGRESSION'
    "MODEL_TYPE": "ARIMA",

    # Time-Series Windowing Parameters
    "LOOK_BACK": 60,                        # Input window
    "TARGET_STEPS": 8,                      # Multi-step Horizon

    # Data Splitting Hyperparameters
    "TRAIN_SPLIT": 0.80,                    # 80% Training, 20% Testing (Chronological split)

    # Feature Engineering Parameters
    "FEATURES": ["Open", "High", "Low", "Close", "Volume"],
    "TARGET_FEATURE": "Close",              # The specific metric to forecast

    # Thesis Artifacts Directory
    "OUTPUT_DIR": "./thesis_results"
}

# Ensure directory exists for saving publication graphs & CSV metrics
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

# Apply a clean, standard styling context for high-resolution publication-ready figures
plt.style.use('default')

print(f"Cell 2 executed successfully: Configuration initialized for {CONFIG['MODEL_TYPE']} model.")

In [ ]:
# [CELL 3] EXPLORATORY DATA ANALYSIS (EDA) ENGINE
def run_publication_eda(df, config):
    """
    Generates high-resolution, publication-ready statistical visualizations
    and saves core metrics into the designated output directory.
    """
    print("## Running Dataset Structural Diagnostics...")
    print(f"Shape of Dataset: {df.shape}")
    print("\nData Types & Info:")
    df.info()

    # 1. Statistical Summary (Saved as CSV for Thesis Tables)
    summary = df[config["FEATURES"]].describe()
    summary_path = os.path.join(config["OUTPUT_DIR"], "statistical_summary.csv")
    summary.to_csv(summary_path)
    print(f"\n Statistical summary saved to: {summary_path}")

    # 2. Correlation Matrix Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[config["FEATURES"]].corr(), annot=True, cmap="coolwarm", fmt=".4f", square=True)
    plt.title("Feature Correlation Matrix", fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_correlation_matrix.png"), dpi=300)
    plt.show()

    # 3. Macro Trends: Closing Price & Volume Over Time
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Target Feature (Close Price)
    axes[0].plot(df.index, df[config["TARGET_FEATURE"]], color='#1f77b4', linewidth=1)
    axes[0].set_title("Bitcoin Historical Closing Price Trend", fontweight='bold', fontsize=12)
    axes[0].set_ylabel("Price (USDT)")
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # Trading Volume
    axes[1].fill_between(df.index, df["Volume"], color='#ff7f0e', alpha=0.5)
    axes[1].set_title("Historical Trading Volume", fontweight='bold', fontsize=12)
    axes[1].set_ylabel("Volume")
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_macro_trends.png"), dpi=300)
    plt.show()

    # 4. Feature Distributions Histograms
    df[config["FEATURES"]].hist(bins=50, figsize=(14, 9), color='darkblue', grid=True, edgecolor='black', alpha=0.7)
    plt.suptitle("Feature Distributions Analysis", fontweight='bold', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(config["OUTPUT_DIR"], "eda_feature_distributions.png"), dpi=300)
    plt.show()

print("Cell 3 executed successfully: Publication-ready EDA engine defined.")

In [ ]:
# [CELL 4] DATA PREPARATION & MULTI-STEP SEQUENCE ENGINE
def load_and_index_dataset(config):
    """
    Loads the financial time-series data and establishes a clean
    DatetimeIndex to handle rigorous chronological ordering.
    """
    df = pd.read_csv(config["FILE_PATH"])
    date_col = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    if date_col:
        df[date_col[0]] = pd.to_datetime(df[date_col[0]], format='mixed')
        df.set_index(date_col[0], inplace=True)
    return df

def generate_sequences(data, look_back, target_idx, target_steps):
    """
    Generates input-output windows.
    y captures a continuous vector of 'target_steps' ahead.
    """
    X, y = [], []
    for i in range(len(data) - look_back - target_steps + 1):
        X.append(data[i : (i + look_back), :])
        y.append(data[(i + look_back) : (i + look_back + target_steps), target_idx])

    return np.array(X), np.array(y)

def prep_thesis_data(df, config):
    """
    Transforms raw dataframe into normalized training and testing tensors
    while strictly avoiding forward-looking/data leakage bias.
    """
    data_matrix = df[config["FEATURES"]].values
    target_idx = config["FEATURES"].index(config["TARGET_FEATURE"])

    # Chronological training/testing split (No random shuffling)
    split_boundary = int(len(data_matrix) * config["TRAIN_SPLIT"])
    train_data = data_matrix[:split_boundary]
    test_data = data_matrix[split_boundary:]

    # Fit scaler ONLY on training data
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_train = scaler.fit_transform(train_data)
    scaled_test = scaler.transform(test_data)

    X_train, y_train = generate_sequences(scaled_train, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])
    X_test, y_test = generate_sequences(scaled_test, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])

    return X_train, y_train, X_test, y_test, scaler, target_idx

print("Cell 4 executed successfully: Multi-step data preparation pipeline fully optimized.")

In [ ]:
# [CELL 5] TRADITIONAL MODEL CONFIGURATOR
def configure_traditional_model(model_type):
    """
    Validates the configuration for traditional statistical baselines.
    Unlike deep learning, these architectures either fit in one pass (LR)
    or require localized rolling fits during inference (ARIMA).
    """
    model_type = model_type.upper()

    if model_type not in ["LINEAR_REGRESSION", "ARIMA"]:
        raise ValueError(f" Model Type '{model_type}' is invalid. Use 'LINEAR_REGRESSION' or 'ARIMA'.")

    print(f" Framework validated for {model_type}. Proceeding to cross-validation and training.")

configure_traditional_model(CONFIG["MODEL_TYPE"])

In [ ]:
# [CELL 6] VALIDATION & EVALUATION METRICS ENGINE
def execute_walk_forward_validation(df, config, folds=3):
    """
    Executes Walk-Forward Time-Series Cross-Validation for statistical models.
    Adapts dimensionality automatically for LR or Univariate ARIMA.
    """
    print("## Running Walk-Forward Time-Series Cross Validation...")
    data_matrix = df[config["FEATURES"]].values
    target_idx = config["FEATURES"].index(config["TARGET_FEATURE"])
    fold_size = len(data_matrix) // (folds + 1)
    fold_scores = []

    for f in range(folds):
        train_end = fold_size * (f + 1)
        test_end = train_end + fold_size

        train_block = data_matrix[:train_end]
        test_block = data_matrix[train_end:test_end]

        fold_scaler = MinMaxScaler()
        scaled_train = fold_scaler.fit_transform(train_block)
        scaled_test = fold_scaler.transform(test_block)

        X_tr, y_tr = generate_sequences(scaled_train, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])
        X_te, y_te = generate_sequences(scaled_test, config["LOOK_BACK"], target_idx, config["TARGET_STEPS"])

        if config["MODEL_TYPE"] == "LINEAR_REGRESSION":
            # Flatten 3D to 2D
            X_tr_flat = X_tr.reshape(X_tr.shape[0], -1)
            X_te_flat = X_te.reshape(X_te.shape[0], -1)
            fold_model = LinearRegression()
            fold_model.fit(X_tr_flat, y_tr)
            preds = fold_model.predict(X_te_flat)

        elif config["MODEL_TYPE"] == "ARIMA":
            # Very simplified CV proxy for ARIMA due to extreme computation time
            # of rolling multi-step fits across thousands of test sequences
            continuous_target = np.concatenate([X_tr[0, :, target_idx], y_tr[:, 0]])
            try:
                auto_model = pm.auto_arima(continuous_target, start_p=1, start_q=1, max_p=3, max_q=3, stepwise=True, suppress_warnings=True)
                opt_order = auto_model.order
            except:
                opt_order = (1, 1, 1) # Fallback if auto-arima fails in CV fold

            preds = np.zeros_like(y_te)
            # Evaluate on a sampled subset of the fold to save hours of processing during CV
            eval_indices = range(0, len(X_te), max(1, len(X_te)//10))
            for i in eval_indices:
                hist_window = X_te[i, :, target_idx]
                loc_arima = ARIMA(hist_window, order=opt_order).fit(method='innovations_mle')
                preds[i, :] = loc_arima.forecast(steps=config["TARGET_STEPS"])

            # Mask out un-evaluated indices for the MSE score
            y_te = y_te[eval_indices]
            preds = preds[eval_indices]

        fold_mse = mean_squared_error(y_te, preds)
        fold_scores.append(fold_mse)
        print(f"Fold {f+1}/{folds} Normalized Multi-step MSE: {fold_mse:.6f}")

    return np.mean(fold_scores)

print("Cell 6 executed successfully: Walk-forward validation adapted for Statistical Baselines.")

In [ ]:
# [CELL 7] LOAD DATA & RUN EDA
try:
    raw_df = load_and_index_dataset(CONFIG)
    print(" Dataset loaded successfully from specified path.")
except FileNotFoundError:
    print(f"Dataset not found at {CONFIG['FILE_PATH']}. Generating synthetic dummy data for pipeline testing.")
    dummy_dates = pd.date_range(start="2021-01-01", periods=5000, freq="5min")
    raw_df = pd.DataFrame(np.random.randn(5000, 5), columns=CONFIG["FEATURES"], index=dummy_dates)
    raw_df["Close"] = 50000 + raw_df["Close"].cumsum() * 10

run_publication_eda(raw_df, CONFIG)
print(" Cell 7 executed successfully: Data pipeline ingestion and EDA diagnostics completed.")

In [ ]:
# [CELL 8] CROSS-VALIDATION EXECUTION & MAIN TENSOR TRAIN-TEST SPLIT
print("==============================================================")
print("     PHASE 1: EXECUTING TIME-SERIES CROSS-VALIDATION          ")
print("==============================================================")

mean_cv_loss = execute_walk_forward_validation(raw_df, CONFIG, folds=3)
print(f"\n Overall Mean Cross-Validation MSE: {mean_cv_loss:.6f}")

print("\n==============================================================")
print("     PHASE 2: PREPARING PRODUCTION TRAIN-TEST TENSORS        ")
print("==============================================================")

X_train, y_train, X_test, y_test, scaler, target_idx = prep_thesis_data(raw_df, CONFIG)

print("\n Verification of Multi-Step Tensor Structural Dimensions:")
print(f"• X_train shape (Samples, Lookback, Features): {X_train.shape}")
print(f"• y_train shape (Samples, Target Steps):       {y_train.shape}")
print(f"• X_test shape  (Samples, Lookback, Features): {X_test.shape}")
print(f"• y_test shape  (Samples, Target Steps):       {y_test.shape}")

print("\n Cell 8 executed successfully: Dataset split validated.")

In [ ]:
# [CELL 9] TRADITIONAL MODEL TRAINING ENGINE
print("==============================================================")
print(f"     PHASE 3: EXECUTING {CONFIG['MODEL_TYPE']} PIPELINE")
print("==============================================================")

start_train_time = time.time()

if CONFIG["MODEL_TYPE"] == "LINEAR_REGRESSION":
    print(" Flattening 3D input tensors to 2D for Multiple Linear Regression...")
    X_train_flat = X_train.reshape(X_train.shape[0], -1)

    print(" Training Multiple Linear Regression model...")
    model = LinearRegression()
    model.fit(X_train_flat, y_train)

    optimal_order = None
    total_model_params = X_train_flat.shape[1] * CONFIG["TARGET_STEPS"] + CONFIG["TARGET_STEPS"]

elif CONFIG["MODEL_TYPE"] == "ARIMA":
    print(" Isolating univariate target feature for ARIMA modeling...")
    target_idx = CONFIG["FEATURES"].index(CONFIG["TARGET_FEATURE"])

    # Reconstruct the continuous 1D training series from the sequenced sliding windows
    continuous_train_target = np.concatenate([X_train[0, :, target_idx], y_train[:, 0]])

    print(" Executing Auto-ARIMA to discover optimal (p, d, q) order...")
    auto_model = pm.auto_arima(
        continuous_train_target,
        start_p=1, start_q=1,
        max_p=5, max_q=5,
        seasonal=False,
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        trace=True
    )

    optimal_order = auto_model.order
    print(f"\n Optimal ARIMA Order Discovered: {optimal_order}")
    total_model_params = sum(optimal_order)
    model = None # ARIMA utilizes rolling fit during the evaluation phase

training_duration = time.time() - start_train_time
print(f"\n Training phase finalized in: {training_duration:.2f} seconds.")

In [ ]:
# [CELL 10] FULL 13-ELEMENT EVALUATION & THESIS REPORTING ENGINE
print("==============================================================")
print("     PHASE 4: COMPLETE SYSTEM EVALUATION & PERFORMANCE        ")
print("==============================================================")

start_inf_time = time.time()
scaled_preds = np.zeros_like(y_test)

if CONFIG["MODEL_TYPE"] == "LINEAR_REGRESSION":
    print(" Generating multi-step predictions using Linear Regression...")
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    scaled_preds = model.predict(X_test_flat)

elif CONFIG["MODEL_TYPE"] == "ARIMA":
    print(f" Generating rolling multi-step forecasts using ARIMA{optimal_order}...")
    print(" (Note: Rolling fits evaluate every individual sliding window natively.)")

    target_idx = CONFIG["FEATURES"].index(CONFIG["TARGET_FEATURE"])

    for i in range(len(X_test)):
        historical_window = X_test[i, :, target_idx]

        local_arima = ARIMA(historical_window, order=optimal_order)
        local_arima_fit = local_arima.fit(method='innovations_mle')

        window_forecast = local_arima_fit.forecast(steps=CONFIG["TARGET_STEPS"])
        scaled_preds[i, :] = window_forecast

        if (i + 1) % 500 == 0:
            print(f"   Processed {i + 1} / {len(X_test)} windows...")

inference_duration = time.time() - start_inf_time

# -------------------------------------------------------------------------
# EXACT REPLICATION OF EVALUATION METRICS
# -------------------------------------------------------------------------
num_features = len(CONFIG["FEATURES"])
target_idx = CONFIG["FEATURES"].index(CONFIG["TARGET_FEATURE"])

print("\n Reversing min-max normalization to obtain actual prices (USDT)...")
def inverse_transform_multistep(scaled_matrix, scaler, target_idx, num_features):
    samples, steps = scaled_matrix.shape
    flat_scaled = scaled_matrix.reshape(-1, 1)
    dummy_matrix = np.zeros((flat_scaled.shape[0], num_features))
    dummy_matrix[:, target_idx] = flat_scaled[:, 0]
    inv_flat = scaler.inverse_transform(dummy_matrix)[:, target_idx]
    return inv_flat.reshape(samples, steps)

actual_prices = inverse_transform_multistep(y_test, scaler, target_idx, num_features)
predicted_prices = inverse_transform_multistep(scaled_preds, scaler, target_idx, num_features)

actual_flat = actual_prices.flatten()
predicted_flat = predicted_prices.flatten()

print("\n Computing validation errors...")
mse = mean_squared_error(actual_flat, predicted_flat)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual_flat, predicted_flat)
median_ae = median_absolute_error(actual_flat, predicted_flat)
max_error = np.max(np.abs(actual_flat - predicted_flat))

mape = np.mean(np.abs((actual_flat - predicted_flat) / np.where(actual_flat == 0, 1e-5, actual_flat))) * 100
smape = 200 * np.mean(np.abs(predicted_flat - actual_flat) / (np.abs(actual_flat) + np.abs(predicted_flat) + 1e-5))
r2 = r2_score(actual_flat, predicted_flat)

# Directional Accuracy
last_known_scaled = X_test[:, -1, target_idx]
dummy = np.zeros((len(last_known_scaled), num_features))
dummy[:, target_idx] = last_known_scaled
last_known_actual = scaler.inverse_transform(dummy)[:, target_idx]

step_das = []
for step in range(CONFIG["TARGET_STEPS"]):
    true_dir = np.sign(actual_prices[:, step] - last_known_actual)
    pred_dir = np.sign(predicted_prices[:, step] - last_known_actual)
    step_da = np.mean(true_dir == pred_dir) * 100
    step_das.append(step_da)

directional_accuracy = np.mean(step_das)

master_thesis_report = {
    "MSE": mse,
    "RMSE": rmse,
    "MAE": mae,
    "Median_AE": median_ae,
    "Max_Error": max_error,
    "MAPE(%)": mape,
    "SMAPE(%)": smape,
    "R2_Score": r2,
    "Directional_Accuracy(%)": directional_accuracy,
    "Training_Time(s)": training_duration,
    "Inference_Time(s)": inference_duration,
    "Total_Model_Parameters": total_model_params,
    "Estimated_Model_Size(MB)": 0.0
}

print("\n" + "="*55)
print(f"   MASTER THESIS BENCHMARK REPORT: {CONFIG['MODEL_TYPE']} ENGINE")
print("="*55)
print(f" {'METRIC ELEMENT':<30} | {'VALUE / SCORE':<20}")
print("-" * 55)
for metric_key, value in master_thesis_report.items():
    if "%" in metric_key:
        print(f" • {metric_key:<28} | {value:.4f}%")
    elif "Time" in metric_key:
        print(f" • {metric_key:<28} | {value:.2f} seconds")
    elif "Parameters" in metric_key:
        print(f" • {metric_key:<28} | {int(value):,}")
    elif "Size" in metric_key:
        print(f" • {metric_key:<28} | N/A (Statistical Model)")
    else:
        print(f" • {metric_key:<28} | {value:.6f}")
print("======================================================")

# Export Unified Results to CSV
report_df = pd.DataFrame(list(master_thesis_report.items()), columns=["Thesis Metric Element", "Value Score"])
csv_save_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_complete_thesis_metrics.csv")
report_df.to_csv(csv_save_path, index=False)
print(f" Comprehensive metrics sheet successfully saved to: {csv_save_path}")

# Plot Forecast Comparison Chart
plt.figure(figsize=(14, 6), dpi=300)
plot_tail = 150
plt.plot(actual_flat[-plot_tail:], label="Actual BTC Price (USDT)", color="#1f77b4", linewidth=2)
plt.plot(predicted_flat[-plot_tail:], label=f"Predicted BTC Price ({CONFIG['MODEL_TYPE']})",
         color="#d62728", linestyle="--", linewidth=1.8)

plt.title(f"Bitcoin Price Tracking Analysis ({CONFIG['MODEL_TYPE']} Multi-Step Framework)",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Continuous Multi-Step Timeline Slices", fontsize=11, labelpad=8)
plt.ylabel("Price (USDT)", fontsize=11, labelpad=8)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", fontsize=10, frameon=True)
plt.tight_layout()

chart_save_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['MODEL_TYPE']}_prediction_plot.png")
plt.savefig(chart_save_path, dpi=300)
plt.show()

print(f"\n Cell 10 executed successfully: All 13 metrics captured for {CONFIG['MODEL_TYPE']}.")